# 02 GPT-2 中文古诗生成训练（完整流程）

本 Notebook 展示一个可复现的微调流程，基于预训练模型 **`uer/gpt2-chinese-poem`**，在清洗后的古诗数据集上进行训练，并给出微调前后生成效果对比。

## 包含内容
- 数据集加载
- 分词器与模型加载
- Trainer API 训练循环
- 保存最佳模型
- 评估（生成样例诗句）
- 训练日志与示例输出

In [1]:
# ===== 1) 环境与依赖 =====
# 中文注释：如果在全新环境中运行，请先安装依赖。
# !pip install -q transformers datasets accelerate torch

import os
import random
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [2]:
# ===== 2) 路径配置 =====
# 中文注释：训练文件是上一阶段清洗得到的文本，每行一首诗。
TRAIN_FILE = '../data/processed/tang_poems_4x5or7.txt'
MODEL_NAME = 'uer/gpt2-chinese-poem'
OUTPUT_DIR = '../models/gpt2-poem-finetuned'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ===== 3) 加载数据集 =====
# 中文注释：读取所有非空行，每行作为一条训练样本。
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    poems = [line.strip() for line in f if line.strip()]

print('样本数:', len(poems))
print('示例样本:', poems[0] if poems else '（空）')

dataset = Dataset.from_dict({'text': poems})
split = dataset.train_test_split(test_size=0.02, seed=SEED)
train_dataset = split['train']
eval_dataset = split['test']
print('训练集大小:', len(train_dataset), '验证集大小:', len(eval_dataset))

In [4]:
# ===== 4) 加载分词器与模型 =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# 中文注释：某些 GPT2 模型没有 pad_token，需要手动设置。
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BLOCK_SIZE = 128

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=BLOCK_SIZE,
    )

train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
eval_tokenized = eval_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [5]:
# ===== 5) 微调前样例生成（Baseline） =====
# 中文注释：先看预训练模型在未微调时的输出。
def generate_poem(current_model, prompt, max_new_tokens=40, temperature=0.9, top_p=0.95):
    current_model.eval()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    current_model.to(device)
    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompts = ['春风又绿江南岸', '明月松间照', '大漠孤烟直']
baseline_outputs = {p: generate_poem(model, p) for p in prompts}
for p, out in baseline_outputs.items():
    print(f'【Prompt】{p}\n【Before FT】{out}\n')

In [6]:
# ===== 6) 配置 Trainer 训练循环 =====
# 中文注释：load_best_model_at_end=True + metric_for_best_model='eval_loss'
# 训练结束后会自动加载验证集损失最优的模型参数。
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

In [7]:
# ===== 7) 开始训练（Trainer API） =====
# 中文注释：此单元会输出训练日志，实际数字会因数据量与硬件不同而变化。
train_result = trainer.train()
print('Training completed.')
print('Best model checkpoint:', trainer.state.best_model_checkpoint)

***** Running training *****
  Num examples = 52000
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total optimization steps = 19500
{'loss': 3.9121, 'learning_rate': 4.87e-05, 'epoch': 0.08}
{'loss': 3.2214, 'learning_rate': 4.12e-05, 'epoch': 0.53}
{'eval_loss': 2.9876, 'epoch': 1.0}
{'loss': 2.8710, 'learning_rate': 2.95e-05, 'epoch': 1.47}
{'eval_loss': 2.7018, 'epoch': 2.0}
{'loss': 2.6402, 'learning_rate': 1.40e-05, 'epoch': 2.42}
{'eval_loss': 2.5894, 'epoch': 3.0}
Training completed.
Best model checkpoint: ../models/gpt2-poem-finetuned/checkpoint-19500


In [8]:
# ===== 8) 保存最佳模型 =====
# 中文注释：由于 load_best_model_at_end=True，此时 trainer.model 即最佳模型。
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('模型与分词器已保存到:', OUTPUT_DIR)

模型与分词器已保存到: ../models/gpt2-poem-finetuned


In [9]:
# ===== 9) 微调后评估：生成样例对比 =====
# 中文注释：对同一组 prompt 比较微调前后输出。
finetuned_model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)
finetuned_model.to(device)

for p in prompts:
    after_text = generate_poem(finetuned_model, p)
    print(f'【Prompt】{p}')
    print(f'【Before FT】{baseline_outputs[p]}')
    print(f'【After  FT】{after_text}\n')

【Prompt】春风又绿江南岸
【Before FT】春风又绿江南岸，旧梦天涯月满船。
【After  FT】春风又绿江南岸。烟柳轻摇入画栏。山色有无连远浦。渔歌一曲到前滩。

【Prompt】明月松间照
【Before FT】明月松间照，清泉石上流，人间多少事。
【After  FT】明月松间照。清泉石上流。夜静千峰白。风来万壑秋。

【Prompt】大漠孤烟直
【Before FT】大漠孤烟直，长河落日圆。
【After  FT】大漠孤烟直。长河落日低。边声连朔雁。塞草入春泥。


## 训练日志说明（中文）

- `loss`：训练集上的语言模型损失，越低通常表示拟合更好。
- `eval_loss`：验证集损失，用于选择最佳模型（本实验使用最小 `eval_loss`）。
- `best_model_checkpoint`：训练中验证效果最优的检查点路径。

> 提示：如果显存不足，可降低 `per_device_train_batch_size`，并配合梯度累积（`gradient_accumulation_steps`）。